In [0]:
USE CATALOG gov_transparencia;
USE SCHEMA gold;

CREATE OR REPLACE TABLE gold_comportamental_estabelecimentos
USING DELTA
AS 

SELECT 
    XXHASH64(TRIM(UPPER(cartao_corporativo.estabelecimento_nome))) AS id_hash_estabelecimento,
    cartao_corporativo.estabelecimento_nome,
    cartao_corporativo.estabelecimento_id,
    cartao_corporativo.estabelecimento_cnpj,
    cartao_corporativo.estabelecimento_tipo,
    CAST(SUM(cartao_corporativo.valor_transacao) AS DECIMAL(18,2)) AS total_transacionado,
    COUNT(cartao_corporativo.id) as qtd_transacoes,
    CAST(AVG(cartao_corporativo.valor_transacao) AS DECIMAL(18,2)) AS valor_medio_p_transacao,
    CAST(STDDEV(cartao_corporativo.valor_transacao) AS DECIMAL(18,2)) AS desvio_padrao_transacao,
    CAST((STDDEV(cartao_corporativo.valor_transacao) / NULLIF(AVG(cartao_corporativo.valor_transacao), 0)) * 100 AS DECIMAL(10,2)) AS cv_percentual,
    DATEDIFF(CURRENT_DATE(),MAX(data_transacao)) AS recencia_dias,
    CAST(COUNT(id) / COUNT(DISTINCT DATE_TRUNC('month', data_transacao)) AS DECIMAL(18,2)) AS freq_media_mensal

FROM gov_transparencia.silver.cartao_corporativo

GROUP BY estabelecimento_nome, estabelecimento_id, estabelecimento_cnpj, estabelecimento_tipo